In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os
import shutil

PROJECT_DIR = "/content/drive/MyDrive/final-year-research"
WORK_DIR = "/content/final-year-research"

SRC_FOLDER = f"{PROJECT_DIR}/datasets/gaussian_faceblur_dataset"
DEST_FOLDER = f"{WORK_DIR}/gaussian_faceblur_dataset"

os.makedirs(WORK_DIR, exist_ok=True)
if os.path.exists(DEST_FOLDER):
    shutil.rmtree(DEST_FOLDER)   # delete existing folder

shutil.copytree(SRC_FOLDER, DEST_FOLDER)

print("Folder copied successfully")

Folder copied successfully


In [3]:
!find /content/final-year-research -maxdepth 3 -type d

/content/final-year-research
/content/final-year-research/gaussian_faceblur_dataset
/content/final-year-research/gaussian_faceblur_dataset/valid
/content/final-year-research/gaussian_faceblur_dataset/valid/images
/content/final-year-research/gaussian_faceblur_dataset/valid/labels
/content/final-year-research/gaussian_faceblur_dataset/train
/content/final-year-research/gaussian_faceblur_dataset/train/images
/content/final-year-research/gaussian_faceblur_dataset/train/labels


In [4]:
!ls /content/final-year-research/gaussian_faceblur_dataset

coco.train.json  data.yaml	     README.roboflow.txt  valid
coco.valid.json  README.dataset.txt  train


In [ ]:
import glob

ROOT = "/content/final-year-research/gaussian_faceblur_dataset"

print("train images:", len(glob.glob(ROOT + "/train/images/*")))
print("train labels:", len(glob.glob(ROOT + "/train/labels/*")))
print("valid images:", len(glob.glob(ROOT + "/valid/images/*")))
print("valid labels:", len(glob.glob(ROOT + "/valid/labels/*")))

train images: 10098
train labels: 10098
valid images: 3284
valid labels: 3284


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: Tesla T4


In [ ]:
!pip install transformers pycocotools -q

In [ ]:
import os

BASE = "/content/final-year-research"

for root, dirs, files in os.walk(BASE):
    for file in files:
        if file.endswith(".json"):
            print(os.path.join(root, file))

/content/final-year-research/gaussian_faceblur_dataset/coco.train.json
/content/final-year-research/gaussian_faceblur_dataset/coco.valid.json


In [ ]:
import os, json, torch, random
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForObjectDetection, AutoImageProcessor
from PIL import Image

# ── Config ───────────────────────────────────────────────
EXPERIMENT   = "gaussian_blur_wilderperson"
DATASET      = "gaussian_faceblur_dataset"
BASE         = "/content/final-year-research"
DRIVE_OUT    = "/content/drive/MyDrive/final-year-research/runs"

MODEL_NAME   = "PekingU/rtdetr_r18vd"
EPOCHS       = 60
BATCH_SIZE   = 16
LR           = 2e-5
SUBSET_FRAC  = 0.6
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

IMAGES_TRAIN = f"{BASE}/{DATASET}/train/images"
IMAGES_VAL   = f"{BASE}/{DATASET}/valid/images"
TRAIN_JSON   = f"{BASE}/{DATASET}/coco.train.json"
VAL_JSON     = f"{BASE}/{DATASET}/coco.valid.json"
SAVE_PATH    = f"{DRIVE_OUT}/rtdetr_{EXPERIMENT}"
# ────────────────────────────────────────────────────────

os.makedirs(f"{SAVE_PATH}/best", exist_ok=True)
os.makedirs(f"{SAVE_PATH}/last", exist_ok=True)

ID2LABEL = {0: "person"}
LABEL2ID = {"person": 0}

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

model = AutoModelForObjectDetection.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
).to(DEVICE)

print(f"✓ Model loaded on {DEVICE}")


class COCODataset(Dataset):
    def __init__(self, images_dir, json_path, processor, subset_frac=1.0):
        self.images_dir = images_dir
        self.processor = processor

        with open(json_path) as f:
            coco = json.load(f)

        # keep only person annotations: original category_id 2
        person_anns = []
        for ann in coco["annotations"]:
            if ann["category_id"] == 2:
                ann = ann.copy()
                ann["category_id"] = 0
                person_anns.append(ann)

        used_image_ids = set(ann["image_id"] for ann in person_anns)

        self.images = {
            img["id"]: img for img in coco["images"]
            if img["id"] in used_image_ids
        }

        self.img_ids = list(self.images.keys())

        if subset_frac < 1.0:
            random.shuffle(self.img_ids)
            keep_n = int(len(self.img_ids) * subset_frac)
            self.img_ids = self.img_ids[:keep_n]

        self.anns = {img_id: [] for img_id in self.img_ids}

        for ann in person_anns:
            if ann["image_id"] in self.anns:
                self.anns[ann["image_id"]].append(ann)

        print("Person-only images:", len(self.img_ids))
        print("Person-only annotations:", sum(len(v) for v in self.anns.values()))

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.images[img_id]

        image = Image.open(
            os.path.join(self.images_dir, img_info["file_name"])
        ).convert("RGB")

        anns = self.anns[img_id]

        target = {
            "image_id": img_id,
            "annotations": [
                {
                    "bbox": a["bbox"],
                    "category_id": a["category_id"],  # now 0 = person
                    "iscrowd": 0,
                    "area": a["bbox"][2] * a["bbox"][3]
                }
                for a in anns
            ]
        }

        encoding = self.processor(
            images=image,
            annotations=target,
            return_tensors="pt"
        )

        return {
            "pixel_values": encoding["pixel_values"].squeeze(0),
            "labels": encoding["labels"][0]
        }


def collate_fn(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "labels": [b["labels"] for b in batch]
    }


train_ds = COCODataset(
    IMAGES_TRAIN,
    TRAIN_JSON,
    processor,
    subset_frac=SUBSET_FRAC
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

print(f"✓ {len(train_ds)} person-only training images loaded")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

best_loss = float("inf")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for i, batch in enumerate(train_loader):
        pixel_values = batch["pixel_values"].to(DEVICE)
        labels = [
            {k: v.to(DEVICE) for k, v in t.items()}
            for t in batch["labels"]
        ]

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()

        total_loss += loss.item()

        if i % 50 == 0:
            print(
                f"  Epoch {epoch+1} | Step {i}/{len(train_loader)} | Loss: {loss.item():.4f}"
            )

    avg_loss = total_loss / len(train_loader)
    scheduler.step()

    print(f"Epoch {epoch+1}/{EPOCHS} — Avg Loss: {avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        model.save_pretrained(f"{SAVE_PATH}/best")
        processor.save_pretrained(f"{SAVE_PATH}/best")
        print(f"  ✓ Best model saved (loss: {best_loss:.4f})")

model.save_pretrained(f"{SAVE_PATH}/last")
processor.save_pretrained(f"{SAVE_PATH}/last")

print("✓ Person-only training complete.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

[transformers] You passed `num_labels=1` which is incompatible to the `id2label` map of length `80`.


model.safetensors:   0%|          | 0.00/80.9M [00:00<?, ?B/s]

...

  ✓ Best model saved (loss: 6.1299)
  Epoch 18 | Step 0/379 | Loss: 6.4231
  Epoch 18 | Step 50/379 | Loss: 5.7889
  Epoch 18 | Step 100/379 | Loss: 6.1785
  Epoch 18 | Step 150/379 | Loss: 6.0798
  Epoch 18 | Step 200/379 | Loss: 5.8763
  Epoch 18 | Step 250/379 | Loss: 5.8661
  Epoch 18 | Step 300/379 | Loss: 5.5125
  Epoch 18 | Step 350/379 | Loss: 5.9898
Epoch 18/60 — Avg Loss: 6.0895


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved (loss: 6.0895)
  Epoch 19 | Step 0/379 | Loss: 6.2979
  Epoch 19 | Step 50/379 | Loss: 6.1959
  Epoch 19 | Step 100/379 | Loss: 5.4602
  Epoch 19 | Step 150/379 | Loss: 5.9073
  Epoch 19 | Step 200/379 | Loss: 6.4398
  Epoch 19 | Step 250/379 | Loss: 5.7138
  Epoch 19 | Step 300/379 | Loss: 6.1758
  Epoch 19 | Step 350/379 | Loss: 5.8431
Epoch 19/60 — Avg Loss: 6.0534


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved (loss: 6.0534)
  Epoch 20 | Step 0/379 | Loss: 6.2651
  Epoch 20 | Step 50/379 | Loss: 5.9309
  Epoch 20 | Step 100/379 | Loss: 5.7842
  Epoch 20 | Step 150/379 | Loss: 6.4971
  Epoch 20 | Step 200/379 | Loss: 6.3114
  Epoch 20 | Step 250/379 | Loss: 6.3694
  Epoch 20 | Step 300/379 | Loss: 6.0049
  Epoch 20 | Step 350/379 | Loss: 6.1981


In [6]:
import os, json, torch, random
from torch.utils.data import DataLoader, Dataset, Subset
from transformers import AutoModelForObjectDetection, AutoImageProcessor
from PIL import Image

# ── Config ───────────────────────────────────────────────
EXPERIMENT   = "gaussian_blur_wilderperson"
DATASET      = "gaussian_faceblur_dataset"
BASE         = "/content/final-year-research"
DRIVE_OUT    = "/content/drive/MyDrive/final-year-research/runs"

EPOCHS       = 40
BATCH_SIZE   = 16
LR           = 2e-5
SUBSET_FRAC  = 0.6
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"       # use 30% of data — balance speed vs coverage


IMAGES_TRAIN = f"{BASE}/{DATASET}/train/images"
TRAIN_JSON   = f"{BASE}/{DATASET}/coco.train.json"
CHECKPOINT   = f"{DRIVE_OUT}/rtdetr_{EXPERIMENT}/best"   # ← resume from here
SAVE_PATH    = f"{DRIVE_OUT}/rtdetr_{EXPERIMENT}"


CATEGORY_REMAP = {2: 0}
ID2LABEL = {0: "person"}
LABEL2ID = {"person": 0}
# ────────────────────────────────────────────────────────

os.makedirs(f"{SAVE_PATH}/best", exist_ok=True)
os.makedirs(f"{SAVE_PATH}/last", exist_ok=True)

# ── Load from checkpoint ─────────────────────────────────
print("Loading from checkpoint...")
processor = AutoImageProcessor.from_pretrained(CHECKPOINT)
model = AutoModelForObjectDetection.from_pretrained(
    CHECKPOINT,
    num_labels=1,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
).to(DEVICE)
print(f"✓ Checkpoint loaded on {DEVICE}")


class COCODataset(Dataset):
    def __init__(self, images_dir, json_path, processor):
        self.images_dir = images_dir
        self.processor  = processor

        with open(json_path) as f:
            coco = json.load(f)

        # Filter to person only and remap
        person_anns = []
        for ann in coco["annotations"]:
            if ann["category_id"] == 2:
                a = ann.copy()
                a["category_id"] = 0
                person_anns.append(a)

        used_image_ids = set(a["image_id"] for a in person_anns)

        # Only keep images that have person annotations
        self.images  = {
            img["id"]: img for img in coco["images"]
            if img["id"] in used_image_ids      # ← filtered
        }
        self.img_ids = list(self.images.keys())
        self.anns    = {img_id: [] for img_id in self.img_ids}

        for ann in person_anns:                 # ← use filtered anns
            if ann["image_id"] in self.anns:
                self.anns[ann["image_id"]].append(ann)

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id   = self.img_ids[idx]
        img_info = self.images[img_id]
        image    = Image.open(
            os.path.join(self.images_dir, img_info["file_name"])
        ).convert("RGB")

        anns = self.anns[img_id]
        target = {"image_id": img_id, "annotations": [
            {
                "bbox": a["bbox"],
                "category_id": CATEGORY_REMAP.get(a["category_id"], a["category_id"]),
                "iscrowd": 0,
                "area": a["bbox"][2] * a["bbox"][3]
            }
            for a in anns
        ]}

        encoding = self.processor(
            images=image,
            annotations=target,
            return_tensors="pt"
        )

        # ── Fix: only squeeze tensors, skip lists ────────────
        return {
            k: v.squeeze(0) if isinstance(v, torch.Tensor) else v
            for k, v in encoding.items()
        }


def collate_fn(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "labels": [b["labels"] if isinstance(b["labels"], dict)
                   else b["labels"][0] if isinstance(b["labels"], list) and len(b["labels"]) > 0
                   else {}
                   for b in batch]
    }


# ── Subset ───────────────────────────────────────────────
full_ds    = COCODataset(IMAGES_TRAIN, TRAIN_JSON, processor)
subset_idx = random.sample(range(len(full_ds)), int(SUBSET_FRAC * len(full_ds)))
train_ds   = Subset(full_ds, subset_idx)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=collate_fn,
    num_workers=2, pin_memory=True
)

print(f"✓ Using {len(train_ds)}/{len(full_ds)} images ({int(SUBSET_FRAC*100)}% subset)")

# ── Optimizer with warmup + cosine decay ─────────────────
optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
total_steps  = EPOCHS * len(train_loader)
warmup_steps = 2 * len(train_loader)   # 2 epoch warmup

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + torch.cos(torch.tensor(progress * 3.14159)).item())

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ── Training Loop ────────────────────────────────────────
best_loss = float("inf")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for i, batch in enumerate(train_loader):
        pixel_values = batch["pixel_values"].to(DEVICE)
        labels = [{k: v.to(DEVICE) for k, v in t.items()}
                  for t in batch["labels"]]

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss    = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        if i % 50 == 0:
            print(f"  Epoch {epoch+1} | Step {i}/{len(train_loader)} | Loss: {loss.item():.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} — Avg Loss: {avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        model.save_pretrained(f"{SAVE_PATH}/best")
        processor.save_pretrained(f"{SAVE_PATH}/best")
        print(f"  ✓ Best model saved (loss: {best_loss:.4f})")

model.save_pretrained(f"{SAVE_PATH}/last")
processor.save_pretrained(f"{SAVE_PATH}/last")
print("✓ Training complete.")

Loading from checkpoint...


Loading weights:   0%|          | 0/526 [00:00<?, ?it/s]

✓ Checkpoint loaded on cuda
✓ Using 6058/10098 images (60% subset)
  Epoch 1 | Step 0/379 | Loss: 6.0913 | LR: 2.64e-08
  Epoch 1 | Step 50/379 | Loss: 6.2141 | LR: 1.35e-06
  Epoch 1 | Step 100/379 | Loss: 6.3473 | LR: 2.66e-06
  Epoch 1 | Step 150/379 | Loss: 6.6887 | LR: 3.98e-06
  Epoch 1 | Step 200/379 | Loss: 6.7017 | LR: 5.30e-06
  Epoch 1 | Step 250/379 | Loss: 6.3591 | LR: 6.62e-06
  Epoch 1 | Step 300/379 | Loss: 6.4280 | LR: 7.94e-06
  Epoch 1 | Step 350/379 | Loss: 6.1860 | LR: 9.26e-06
Epoch 1/40 — Avg Loss: 6.3159


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved (loss: 6.3159)
  Epoch 2 | Step 0/379 | Loss: 5.9186 | LR: 1.00e-05
  Epoch 2 | Step 50/379 | Loss: 6.6758 | LR: 1.13e-05
  Epoch 2 | Step 100/379 | Loss: 6.2927 | LR: 1.27e-05
  Epoch 2 | Step 150/379 | Loss: 6.5152 | LR: 1.40e-05
  Epoch 2 | Step 200/379 | Loss: 5.9492 | LR: 1.53e-05
  Epoch 2 | Step 250/379 | Loss: 5.7214 | LR: 1.66e-05
  Epoch 2 | Step 300/379 | Loss: 6.2584 | LR: 1.79e-05
  Epoch 2 | Step 350/379 | Loss: 6.1769 | LR: 1.93e-05
Epoch 2/40 — Avg Loss: 6.3151


...

  ✓ Best model saved (loss: 5.0922)
  Epoch 38 | Step 0/379 | Loss: 6.8056 | LR: 3.05e-07
  Epoch 38 | Step 50/379 | Loss: 5.4508 | LR: 2.79e-07
  Epoch 38 | Step 100/379 | Loss: 4.6335 | LR: 2.54e-07
  Epoch 38 | Step 150/379 | Loss: 4.7930 | LR: 2.30e-07
  Epoch 38 | Step 200/379 | Loss: 4.9491 | LR: 2.08e-07
  Epoch 38 | Step 250/379 | Loss: 4.5744 | LR: 1.86e-07
  Epoch 38 | Step 300/379 | Loss: 4.6439 | LR: 1.66e-07
  Epoch 38 | Step 350/379 | Loss: 4.8183 | LR: 1.47e-07
Epoch 38/40 — Avg Loss: 5.0844


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved (loss: 5.0844)
  Epoch 39 | Step 0/379 | Loss: 4.9571 | LR: 1.36e-07
  Epoch 39 | Step 50/379 | Loss: 4.9910 | LR: 1.19e-07
  Epoch 39 | Step 100/379 | Loss: 4.7242 | LR: 1.03e-07
  Epoch 39 | Step 150/379 | Loss: 4.7445 | LR: 8.75e-08
  Epoch 39 | Step 200/379 | Loss: 5.4684 | LR: 7.37e-08
  Epoch 39 | Step 250/379 | Loss: 5.3786 | LR: 6.11e-08
  Epoch 39 | Step 300/379 | Loss: 4.9228 | LR: 4.97e-08
  Epoch 39 | Step 350/379 | Loss: 4.8162 | LR: 3.94e-08
Epoch 39/40 — Avg Loss: 5.0899
  Epoch 40 | Step 0/379 | Loss: 5.5894 | LR: 3.40e-08
  Epoch 40 | Step 50/379 | Loss: 5.5314 | LR: 2.56e-08
  Epoch 40 | Step 100/379 | Loss: 4.9776 | LR: 1.84e-08
  Epoch 40 | Step 150/379 | Loss: 5.2570 | LR: 1.24e-08
  Epoch 40 | Step 200/379 | Loss: 4.9616 | LR: 7.54e-09
  Epoch 40 | Step 250/379 | Loss: 5.0929 | LR: 3.90e-09
  Epoch 40 | Step 300/379 | Loss: 4.9473 | LR: 1.45e-09
  Epoch 40 | Step 350/379 | Loss: 5.3700 | LR: 1.87e-10
Epoch 40/40 — Avg Loss: 5.1029


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Training complete.


In [13]:
from transformers import AutoModelForObjectDetection, AutoImageProcessor
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import torch, json, os
from PIL import Image

# ── Config ───────────────────────────────────────────────
EXPERIMENT = "gaussian_blur_wilderperson"
DATASET    = "gaussian_faceblur_dataset"

BASE       = "/content/final-year-research"
DRIVE_OUT  = "/content/drive/MyDrive/final-year-research/runs"

SAVE_PATH  = f"{DRIVE_OUT}/rtdetr_{EXPERIMENT}/best"
VAL_JSON   = f"/content/drive/MyDrive/final-year-research/datasets/{DATASET}/coco.valid.json"
VAL_IMG    = f"/content/drive/MyDrive/final-year-research/datasets/{DATASET}/valid/images"

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
# ────────────────────────────────────────────────────────

processor = AutoImageProcessor.from_pretrained(SAVE_PATH)
model = AutoModelForObjectDetection.from_pretrained(SAVE_PATH).to(DEVICE)
model.eval()

with open(VAL_JSON) as f:
    val_data = json.load(f)

coco_gt = COCO(VAL_JSON)
results = []

print(f"Running inference on {len(val_data['images'])} images...")

for idx, img_info in enumerate(val_data["images"]):
    image_path = os.path.join(VAL_IMG, img_info["file_name"])
    image = Image.open(image_path).convert("RGB")

    inputs = processor(images=image, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.tensor([image.size[::-1]]).to(DEVICE)

    preds = processor.post_process_object_detection(
        outputs,
        threshold=0.05,
        target_sizes=target_sizes
    )[0]

    for score, label, box in zip(
        preds["scores"], preds["labels"], preds["boxes"]
    ):
        x1, y1, x2, y2 = box.tolist()

        results.append({
            "image_id": img_info["id"],
            "category_id": int(label.item()),  # 0 = person
            "bbox": [x1, y1, x2 - x1, y2 - y1],
            "score": float(score.item())
        })

    if idx % 500 == 0:
        print(f"  {idx}/{len(val_data['images'])} done...")

if len(results) == 0:
    print("No detections found. Try threshold=0.01")
else:
    print(f"Total detections: {len(results)}")

    coco_dt = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, "bbox")

    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    print("\n── RT-DETR Results Person Only ───")
    print(f"mAP50-95 : {coco_eval.stats[0]:.4f}")
    print(f"mAP50    : {coco_eval.stats[1]:.4f}")
    print(f"Recall   : {coco_eval.stats[8]:.4f}")
    print("──────────────────────────────────")

Loading weights:   0%|          | 0/526 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.38s)
creating index...
index created!
Running inference on 3284 images...
  0/3284 done...
  500/3284 done...
  1000/3284 done...
  1500/3284 done...
  2000/3284 done...
  2500/3284 done...
  3000/3284 done...
Total detections: 148648
Loading and preparing results...
DONE (t=0.79s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.78s).
Accumulating evaluation results...
DONE (t=0.56s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ 

In [5]:
import os

print(os.path.exists("/content/final-year-research/gaussian_faceblur_dataset/coco.valid.json"))

False


Testing

In [2]:
import os

path = "/content/drive/MyDrive/final-year-research/runs/rtdetr_gaussian_blur_wilderperson/best"

print(os.path.exists(path))
print(os.listdir(path))

True
['config.json', 'preprocessor_config.json', 'model.safetensors']


In [3]:
from transformers import AutoModelForObjectDetection, AutoImageProcessor

SAVE_PATH = "/content/drive/MyDrive/final-year-research/runs/rtdetr_gaussian_blur_wilderperson/best"

processor = AutoImageProcessor.from_pretrained(SAVE_PATH)
model = AutoModelForObjectDetection.from_pretrained(SAVE_PATH)

Loading weights:   0%|          | 0/526 [00:00<?, ?it/s]

In [4]:
print(model.config.id2label)
print(model.config.num_labels)

{0: 'person'}
1


In [7]:
from transformers import AutoModelForObjectDetection, AutoImageProcessor
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import torch, json, os
from PIL import Image

# ── Config ───────────────────────────────────────────────
EXPERIMENT = "gaussian_blur_wilderperson"
DATASET    = "gaussian_faceblur_dataset"

BASE       = "/content/final-year-research"
DRIVE_OUT  = "/content/drive/MyDrive/final-year-research/runs"

SAVE_PATH  = f"{DRIVE_OUT}/rtdetr_{EXPERIMENT}/best"
VAL_JSON   = f"/content/drive/MyDrive/final-year-research/datasets/{DATASET}/coco.valid.json"
VAL_IMG    = f"/content/drive/MyDrive/final-year-research/datasets/{DATASET}/valid/images"

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"



with open(VAL_JSON) as f:
    val_data = json.load(f)

print(val_data["categories"][:5])

[{'id': 0, 'name': 'Human', 'supercategory': 'none'}, {'id': 1, 'name': 'head', 'supercategory': 'Human'}, {'id': 2, 'name': 'person', 'supercategory': 'Human'}]


In [8]:
import json

VAL_JSON = "/content/drive/MyDrive/final-year-research/datasets/gaussian_faceblur_dataset/coco.valid.json"
OUT_JSON = "/content/drive/MyDrive/final-year-research/datasets/gaussian_faceblur_dataset/person_only_valid.json"

with open(VAL_JSON, "r") as f:
    coco = json.load(f)

new_annotations = []
new_id = 1

for ann in coco["annotations"]:
    if ann["category_id"] == 2:   # keep only person
        ann = ann.copy()
        ann["id"] = new_id
        ann["category_id"] = 0    # remap person to 0
        new_annotations.append(ann)
        new_id += 1

used_image_ids = set(ann["image_id"] for ann in new_annotations)
new_images = [img for img in coco["images"] if img["id"] in used_image_ids]

person_only = {
    "images": new_images,
    "annotations": new_annotations,
    "categories": [
        {"id": 0, "name": "person", "supercategory": "Human"}
    ]
}

with open(OUT_JSON, "w") as f:
    json.dump(person_only, f)

print("Saved:", OUT_JSON)
print("Images:", len(new_images))
print("Annotations:", len(new_annotations))

Saved: /content/drive/MyDrive/final-year-research/datasets/gaussian_faceblur_dataset/person_only_valid.json
Images: 3284
Annotations: 80095


In [9]:
from transformers import AutoModelForObjectDetection, AutoImageProcessor
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import torch, json, os
from PIL import Image

# ── Config ───────────────────────────────────────────────
EXPERIMENT = "gaussian_blur_wilderperson"
DATASET    = "gaussian_faceblur_dataset"

BASE       = "/content/final-year-research"
DRIVE_OUT  = "/content/drive/MyDrive/final-year-research/runs"

SAVE_PATH  = f"{DRIVE_OUT}/rtdetr_{EXPERIMENT}/best"
VAL_JSON   = f"/content/drive/MyDrive/final-year-research/datasets/{DATASET}/person_only_valid.json"
VAL_IMG    = f"/content/drive/MyDrive/final-year-research/datasets/{DATASET}/valid/images"

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
# ────────────────────────────────────────────────────────

processor = AutoImageProcessor.from_pretrained(SAVE_PATH)
model = AutoModelForObjectDetection.from_pretrained(SAVE_PATH).to(DEVICE)
model.eval()

with open(VAL_JSON) as f:
    val_data = json.load(f)

coco_gt = COCO(VAL_JSON)
results = []

print(f"Running inference on {len(val_data['images'])} images...")

for idx, img_info in enumerate(val_data["images"]):
    image_path = os.path.join(VAL_IMG, img_info["file_name"])
    image = Image.open(image_path).convert("RGB")

    inputs = processor(images=image, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.tensor([image.size[::-1]]).to(DEVICE)

    preds = processor.post_process_object_detection(
        outputs,
        threshold=0.05,
        target_sizes=target_sizes
    )[0]

    for score, label, box in zip(
        preds["scores"], preds["labels"], preds["boxes"]
    ):
        x1, y1, x2, y2 = box.tolist()

        results.append({
            "image_id": img_info["id"],
            "category_id": int(label.item()),  # 0 = person
            "bbox": [x1, y1, x2 - x1, y2 - y1],
            "score": float(score.item())
        })

    if idx % 500 == 0:
        print(f"  {idx}/{len(val_data['images'])} done...")

if len(results) == 0:
    print("No detections found. Try threshold=0.01")
else:
    print(f"Total detections: {len(results)}")

    coco_dt = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, "bbox")

    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    print("\n── RT-DETR Results Person Only ───")
    print(f"mAP50-95 : {coco_eval.stats[0]:.4f}")
    print(f"mAP50    : {coco_eval.stats[1]:.4f}")
    print(f"Recall   : {coco_eval.stats[8]:.4f}")
    print("──────────────────────────────────")

Loading weights:   0%|          | 0/526 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.20s)
creating index...
index created!
Running inference on 3284 images...
  0/3284 done...
  500/3284 done...
  1000/3284 done...
  1500/3284 done...
  2000/3284 done...
  2500/3284 done...
  3000/3284 done...
Total detections: 148648
Loading and preparing results...
DONE (t=0.67s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=78.59s).
Accumulating evaluation results...
DONE (t=1.57s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.372
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.610
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.395
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.157
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.388
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.470
 Average Recall     (AR) @[